# Review a prompt and run a fair experiment

Learn where the summary instructions live, inspect a real example, draft one small change, and compare results while keeping the evaluator fixed.

**Default: offline.** Run every cell to read the current prompt, see a candidate diff, check the bundled summary's structure, and save an experiment draft. No credentials are loaded and no model calls are made. Faithfulness and coverage scores are available only after a deliberate live run or loading genuine saved results.

The bundled material is fictional. Its prewritten summaries were **not generated by the candidate prompt in this notebook**. A local model experiment also does not establish that Zoom's deployed summarizer improved.

## 1. Choose inputs and keep live work off

Use the project's Python environment with the `notebooks` extra installed. Start Jupyter from this repository or its `notebooks` folder. Change the paths below to review your own approved inputs.

For a real prompt test, provide baseline and candidate summaries produced from the **same transcript and supplied context**. The default uses the same prewritten fixture for both: a control that cannot establish a prompt improvement.

In [ ]:
import difflib
import hashlib
import html
import inspect
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import HTML, Markdown, display
from meeting_eval import BatchResult, MeetingSummaryEvaluator
from meeting_eval.comparison import compare_batches
from meeting_eval.diagnostics import check_structure, diagnostic_prompt
from meeting_eval.judge import JUDGE_SYSTEM_INSTRUCTION
from meeting_eval.metrics import COVERAGE_STEPS, CLARITY_STEPS, MeetingFaithfulnessTemplate
from meeting_eval.policy import PRODUCER_PROMPT, TEMPLATE_VERSION, METHODOLOGY_VERSION
from meeting_eval.provenance import evaluator_fingerprint, runtime_metadata
from meeting_eval.reports import write_csv

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "pyproject.toml").is_file() and (path / "src/meeting_eval").is_dir())
RUN_LIVE = False
RUN_GENERATION = False  # Requires RUN_LIVE=True; generates both variants in a separate optional step.
MODEL = None           # Live only: None uses OPENAI_MODEL or the package's documented default.
GENERATOR_MODEL = None # Live only: None uses the same model ID as MODEL.
GENERATOR_LLM = None   # Optional caller-configured ChatOpenAI; remains caller-owned.
MAX_CONCURRENCY = 2
MEETING_ID = "meeting_001"
TRANSCRIPT_PATH = ROOT / "examples/data/transcripts/meeting_001.txt"
CONTEXT_PATH = ROOT / "examples/data/contexts/meeting_001.md"  # Or None if the producer received no context.
BASELINE_SUMMARY_PATH = ROOT / "examples/data/summaries/meeting_001.md"
CANDIDATE_SUMMARY_PATH = BASELINE_SUMMARY_PATH  # Change to your actual candidate summary file.
MANUAL_QA_PRESENT = True # This fixture contains substantive Q&A; review this label for other inputs.
SAVED_BASELINE_JSON = None  # Optional Path to a genuine baseline results.json.
SAVED_CANDIDATE_JSON = None # Optional Path to its compatible candidate results.json.

if os.environ.get("MEETING_EVAL_OFFLINE") == "1" and (RUN_LIVE or RUN_GENERATION):
    raise RuntimeError("The learning runner requires offline mode. Leave both live switches False.")
if RUN_GENERATION and not RUN_LIVE:
    raise ValueError("Generation requires explicit RUN_LIVE=True as well as RUN_GENERATION=True.")
FROZEN_EVALUATOR_FINGERPRINT = evaluator_fingerprint()
print("Mode:", "LIVE opt-in" if RUN_LIVE else "OFFLINE — no credentials or model requests")
print("Producer template:", TEMPLATE_VERSION, "| Evaluation methodology:", METHODOLOGY_VERSION)
print("Evaluator fingerprint:", FROZEN_EVALUATOR_FINGERPRINT)

## 2. Find the instructions that do each job

A **producer prompt** tells the summarizer what to write. The **judge prompts** tell the evaluator how to assess it. Change a copy of the producer prompt for this experiment; keep the evaluator policy and code fixed so the comparison answers one question.

If independent human review later reveals a systematic judge error, treat that as a separate judge-calibration change and establish a new evaluation baseline.

In [ ]:
roles = [
    ("src/meeting_eval/prompts/summary-template-v4.1.md", "Producer", "The exact six-section summary instructions used as this notebook's baseline."),
    ("src/meeting_eval/policy.py", "Shared evaluation contract", "Versions, source authority, attribution, and labeled derived outreach. Keep fixed."),
    ("src/meeting_eval/metrics.py", "Metric judges", "Four faithfulness prompts, coverage steps/rubric, and optional clarity. Keep fixed."),
    ("src/meeting_eval/diagnostics.py", "Diagnostic judge and evidence", "Findings, source quotations, and deterministic structure checks. Keep fixed."),
    ("src/meeting_eval/judge.py", "Judge system message and adapter", "Untrusted-data boundary, strict structured responses, and request usage. Keep fixed."),
    ("src/meeting_eval/provenance.py", "Experiment identity", "Safe resolved settings and a fingerprint covering evaluator code, prompts, and schemas."),
]
display(Markdown("| File | Role | What to review |\n|---|---|---|\n" + "\n".join(
    f"| `{path}` | {role} | {purpose} |" for path, role, purpose in roles)))
print("Every listed source exists:", all((ROOT / path).is_file() for path, _, _ in roles))

### Read the actual producer prompt

This is read from the installed package resource, rather than a separately maintained notebook copy.

In [ ]:
baseline_prompt = PRODUCER_PROMPT
display(Markdown(baseline_prompt))
print("Producer prompt SHA-256:", hashlib.sha256(baseline_prompt.encode("utf-8")).hexdigest())

## 3. Read a source, its separate context, and a prewritten summary

Notice the authority boundary: the transcript records what was said in the meeting. The supplied workshop context can clarify facts and identities, but cannot make a recommendation or discussion occur in that meeting.

In [ ]:
transcript = TRANSCRIPT_PATH.read_text(encoding="utf-8-sig")
provided_context = CONTEXT_PATH.read_text(encoding="utf-8-sig") if CONTEXT_PATH is not None else None
baseline_summary = BASELINE_SUMMARY_PATH.read_text(encoding="utf-8-sig")
candidate_summary = CANDIDATE_SUMMARY_PATH.read_text(encoding="utf-8-sig")

for label, text in (("Transcript", transcript), ("Explicitly provided context", provided_context),
                    ("Prewritten baseline summary", baseline_summary)):
    display(Markdown(f"### {label}"))
    if text is None:
        print("No context was supplied.")
    elif label.endswith("summary"):
        display(Markdown(text))
    else:
        print(text)
print("Prewritten summary inputs are identical:", baseline_summary == candidate_summary)
print("These files have no established generation relationship to the candidate prompt below.")

### Inspect one exact judge instruction

Choose another key below to inspect the claim-extraction source, verdict source, coverage/clarity steps, or system instruction. Opening these views makes no requests. The diagnostic view includes this loaded transcript, context, summary, and the producer contract.

In [ ]:
REVIEW_COMPONENT = "diagnostics_for_loaded_pair"
review_material = {
    "diagnostics_for_loaded_pair": diagnostic_prompt(transcript, baseline_summary, provided_context),
    "faithfulness_claim_extraction_source": inspect.getsource(MeetingFaithfulnessTemplate.generate_claims),
    "faithfulness_verdict_source": inspect.getsource(MeetingFaithfulnessTemplate.generate_verdicts),
    "coverage_steps": "\n\n".join(COVERAGE_STEPS),
    "clarity_steps": "\n\n".join(CLARITY_STEPS),
    "judge_system": JUDGE_SYSTEM_INSTRUCTION,
}
selected_instruction = review_material[REVIEW_COMPONENT]
print("Available views:", ", ".join(review_material))
print("Selected:", REVIEW_COMPONENT, "| Characters:", len(selected_instruction))
display(HTML("<details><summary>Open exact instruction text</summary>"
             "<pre style='white-space:pre-wrap;max-height:32rem;overflow:auto'>"
             + html.escape(selected_instruction) + "</pre></details>"))

## 4. Draft one focused producer change

Example hypothesis: an explicit final number check might reduce mistakes in units and time horizons. The baseline already requires accurate numbers; this candidate adds a concrete review step. It is a proposal, not a demonstrated improvement.

Edit `CANDIDATE_ADDENDUM` and the notes. This changes a separate Python string. It does **not** modify the packaged producer resource, shared policy, or any judge prompt.

In [ ]:
EXPERIMENT_NAME = "check-number-unit-and-horizon"
HYPOTHESIS = "A final source check may reduce numeric distortions without dropping useful qualifications."
SUCCESS_CRITERION = "Human review finds fewer material numeric errors and no new material omissions or attribution errors."
REVIEW_NOTES = "Not reviewed yet."
CANDIDATE_ADDENDUM = """Final accuracy check before returning the six-section summary:
For each market number you include, compare its value, unit, maturity or scope, and time
horizon with the source passage. Correct mismatches and preserve essential qualifications.
Do not add a checking section or describe this review in the returned summary."""
candidate_prompt = baseline_prompt.rstrip() + "\n\n" + CANDIDATE_ADDENDUM.strip() + "\n"
prompt_diff = "\n".join(difflib.unified_diff(
    baseline_prompt.splitlines(), candidate_prompt.splitlines(),
    fromfile="baseline-producer-prompt", tofile="candidate-producer-prompt", lineterm=""))
print(prompt_diff)
assert candidate_prompt != baseline_prompt
assert evaluator_fingerprint() == FROZEN_EVALUATOR_FINGERPRINT
print("The evaluator fingerprint stayed fixed; the candidate is a separate producer prompt.")

### Make predictions before asking a judge

Use the loaded transcript to predict the treatment of these **exercise sentences**. They are not model outputs or scored results. The point is to test whether a proposed prompt change addresses the error that matters.

In [ ]:
display(Markdown("""| Exercise sentence | What should a reviewer check? |
|---|---|
| “Short-term yields declined 10% this year.” | Value, unit, maturity, and horizon against Alex's 01:20 remark. |
| “Fidelity confirmed a rate cut next month.” | A participant question versus Jordan's qualified 06:35 answer. |
| “Suggested outreach: Explore the advisor's near-term liquidity needs.” | A labeled generated question versus a claimed meeting endorsement. |
| “The workshop identifier was CM-TEST-001.” | Separate context can support the identifier; it cannot establish that it was spoken in the meeting. |
"""))
display(HTML("<details><summary>Show the boundary reminders</summary>"
             "<p>Basis points are not percent returns; the cited yield move was weekly. "
             "The source gives no rate-cut date forecast. A labeled exploratory question can be "
             "derived from a supported insight, while an asserted recommendation needs explicit "
             "meeting support. The identifier appears only in provided context.</p></details>"))

## 5. Run the structural checks offline

These are real deterministic checks of headings, item limits, provenance labels, tags, and related template requirements. They do not assess whether a financial claim is supported or whether important meaning was omitted. `MANUAL_QA_PRESENT` above is a human-supplied label for this exercise.

In [ ]:
structure_result, structure_findings = check_structure(baseline_summary, MANUAL_QA_PRESENT)
checks = structure_result.details["checks"]
display(Markdown("| Template check | Result |\n|---|---|\n" + "\n".join(
    f"| `{name}` | {'Not determined' if passed is None else 'Satisfied' if passed else 'Needs attention'} |"
    for name, passed in checks.items())))
print("Executive bullet count:", structure_result.details["executive_bullet_count"])
for finding in structure_findings:
    print(finding.description)
print("Faithfulness: not run. Coverage: not run. Structural checks do not establish summary quality.")

## 6. Save an inspectable experiment draft

The draft saves both exact producer prompts, the diff, input identities, and your hypothesis. It records that the default summaries are prewritten inputs. Local result files can contain source quotations; keep them in an appropriate local location when using real meetings.

In [ ]:
def text_sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest() if text is not None else None

experiment_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
experiment_dir = ROOT / "results/prompt_experiments" / experiment_id
experiment_dir.mkdir(parents=True, exist_ok=False)
(experiment_dir / "baseline_prompt.md").write_text(baseline_prompt, encoding="utf-8")
(experiment_dir / "candidate_prompt.md").write_text(candidate_prompt, encoding="utf-8")
(experiment_dir / "producer_prompt.diff").write_text(prompt_diff + "\n", encoding="utf-8")
manifest = {
    "experiment_id": experiment_id, "name": EXPERIMENT_NAME, "status": "offline_draft_no_quality_scores",
    "hypothesis": HYPOTHESIS, "success_criterion": SUCCESS_CRITERION,
    "human_review": {"status": "pending", "notes": REVIEW_NOTES},
    "evaluator_fingerprint": FROZEN_EVALUATOR_FINGERPRINT,
    "producer_template_version": TEMPLATE_VERSION, "methodology_version": METHODOLOGY_VERSION,
    "meeting_id": MEETING_ID,
    "transcript": {"path": str(TRANSCRIPT_PATH), "sha256": text_sha256(transcript)},
    "provided_context": {"path": str(CONTEXT_PATH) if CONTEXT_PATH is not None else None,
                         "sha256": text_sha256(provided_context)},
    "producer_prompts": {
        "baseline": {"file": "baseline_prompt.md", "sha256": text_sha256(baseline_prompt)},
        "candidate": {"file": "candidate_prompt.md", "sha256": text_sha256(candidate_prompt)},
    },
    "summary_inputs": {
        "baseline": {"path": str(BASELINE_SUMMARY_PATH), "sha256": text_sha256(baseline_summary), "origin": "prewritten_input"},
        "candidate": {"path": str(CANDIDATE_SUMMARY_PATH), "sha256": text_sha256(candidate_summary), "origin": "prewritten_input"},
        "generation_relationship": "Not established by this notebook unless RUN_GENERATION is enabled.",
    },
    "generation": None, "evaluation_runs": None,
    "limits": "Local generation is not the deployed Zoom summarizer; fixture scores do not establish production improvement.",
}
manifest_path = experiment_dir / "experiment_manifest.json"

def save_manifest():
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")

save_manifest()
print("Saved experiment draft:", experiment_dir)
print("Prompt hashes differ:", manifest["producer_prompts"]["baseline"]["sha256"] != manifest["producer_prompts"]["candidate"]["sha256"])
print("Human review:", manifest["human_review"]["status"])

## 7. Optional live generation — off by default

Skip this step to evaluate summaries you already obtained from Zoom or another producer. To run it, deliberately set **both** switches at the top to `True`, configure your approved endpoint/model, and rerun from the top.

This makes **two generator invocations**, one for each exact producer prompt, using the same sources and model settings. Generator retries are disabled in the notebook-created client. A caller-supplied `GENERATOR_LLM` retains its own settings and ownership. The manifest records safe settings, exact prompt hashes, actual response model when available, and provider-reported token usage.

This tests a local provider configuration. Zoom can use different models, source preparation, and instructions, so these results cannot establish an improvement to Zoom itself.

In [ ]:
if RUN_GENERATION:
    if os.environ.get("MEETING_EVAL_OFFLINE") == "1" or not RUN_LIVE:
        raise RuntimeError("Live generation is disabled by the selected execution mode.")
    from dotenv import load_dotenv
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI
    from langsmith import tracing_context
    from openai import DefaultAsyncHttpxClient, DefaultHttpxClient

    load_dotenv(ROOT / ".env", override=False)
    resolved_generator_model = GENERATOR_MODEL or MODEL or os.getenv("OPENAI_MODEL", "gpt-6-luna")
    generator = GENERATOR_LLM
    owns_generator = generator is None
    if owns_generator:
        generator = ChatOpenAI(model=resolved_generator_model, timeout=120, max_retries=0,
                               http_client=DefaultHttpxClient(), http_async_client=DefaultAsyncHttpxClient())
    manifest["generation"] = {"kind": "local_provider_experiment", "is_zoom_production": False, "variants": {}}
    try:
        settings = runtime_metadata(generator)
        manifest["generation"].update({"model": settings["model"], "settings": settings["judge_config"], "versions": settings["versions"]})
        source_message = json.dumps({"transcript": transcript, "provided_context": provided_context}, ensure_ascii=False)
        generated = {}
        for variant, prompt in (("baseline", baseline_prompt), ("candidate", candidate_prompt)):
            with tracing_context(enabled=False):
                response = await generator.ainvoke([SystemMessage(content=prompt), HumanMessage(content=source_message)])
            text = response.text
            if not isinstance(text, str) or not text.strip():
                raise ValueError("The generator returned no summary text.")
            generated[variant] = text
            summary_path = experiment_dir / f"generated_{variant}_summary.md"
            summary_path.write_text(text, encoding="utf-8")
            usage = response.usage_metadata or {}
            manifest["generation"]["variants"][variant] = {
                "prompt_sha256": text_sha256(prompt),
                "response_model": response.response_metadata.get("model_name"),
                "usage": {key: usage.get(key) for key in ("input_tokens", "output_tokens", "total_tokens")},
            }
            manifest["summary_inputs"][variant] = {"path": str(summary_path), "sha256": text_sha256(text), "origin": "generated_in_this_experiment"}
            save_manifest()
        baseline_summary, candidate_summary = generated["baseline"], generated["candidate"]
        manifest["summary_inputs"]["generation_relationship"] = "Both summaries generated with the separately retained exact producer prompts and shared source inputs."
        manifest["status"] = "generated_not_yet_evaluated"
        print("Generated both local variants. Inspect them before interpreting any judge score.")
    except Exception as exc:
        manifest["status"] = "generation_failed"
        manifest["error_type"] = type(exc).__name__
        raise RuntimeError("Generation stopped. Review the saved manifest and your provider configuration before retrying.") from None
    finally:
        save_manifest()
        if owns_generator:
            await generator.root_async_client.close()
            generator.root_client.close()
else:
    print("OFFLINE/default: generation skipped; the loaded prewritten summaries remain the inputs.")

## 8. Optional live evaluation with one frozen judge

Set `RUN_LIVE=True` only when ready to send these source/summary pairs to the configured model endpoint. This evaluates **two pairs**, usually **12 judge invocations** in total: four faithfulness stages, one coverage call, and one diagnostic call per pair, before any SDK retries. Optional generation above adds two invocations. Full transcripts are repeatedly sent, so input length matters to cost and rate limits.

The same evaluator handles both variants. Each run writes its own detailed JSON, CSV, and HTML. A failed baseline stops the exercise before spending on the candidate. Authentication, schema, and timeout problems are execution problems, not bad-summary scores.

With the default identical prewritten summaries, score variation measures judge variation on a control. It cannot be credited to the candidate producer prompt.

In [ ]:
live_baseline = live_candidate = None
if RUN_LIVE:
    if os.environ.get("MEETING_EVAL_OFFLINE") == "1":
        raise RuntimeError("The learning runner requires offline mode.")
    if evaluator_fingerprint() != FROZEN_EVALUATOR_FINGERPRINT:
        raise RuntimeError("The evaluator changed. Restart this experiment and establish a new baseline.")
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=False)
    resolved_judge_model = MODEL or os.getenv("OPENAI_MODEL", "gpt-6-luna")
    inputs_dir = experiment_dir / "evaluation_inputs"
    transcript_dir = inputs_dir / "transcripts"
    baseline_dir, candidate_dir = inputs_dir / "baseline_summaries", inputs_dir / "candidate_summaries"
    context_dir = inputs_dir / "contexts" if provided_context is not None else None
    for folder in (transcript_dir, baseline_dir, candidate_dir, context_dir):
        if folder is not None:
            folder.mkdir(parents=True, exist_ok=True)
    (transcript_dir / f"{MEETING_ID}.txt").write_text(transcript, encoding="utf-8")
    (baseline_dir / f"{MEETING_ID}.md").write_text(baseline_summary, encoding="utf-8")
    (candidate_dir / f"{MEETING_ID}.md").write_text(candidate_summary, encoding="utf-8")
    if context_dir is not None:
        (context_dir / f"{MEETING_ID}.md").write_text(provided_context, encoding="utf-8")
    manifest["status"] = "evaluating"
    manifest["evaluation_runs"] = {}
    save_manifest()
    try:
        async with MeetingSummaryEvaluator(model=resolved_judge_model, max_concurrency=MAX_CONCURRENCY,
                                           metrics=("faithfulness", "coverage"), env_file=None) as evaluator:
            live_baseline = await evaluator.aevaluate_batch(transcript_dir, baseline_dir,
                experiment_dir / "judge_runs", context_dir=context_dir)
            manifest["evaluation_runs"]["baseline"] = str(Path(live_baseline.output_dir) / "results.json")
            save_manifest()
            if any(result.execution_status != "completed" for result in live_baseline.results):
                raise RuntimeError("Baseline evaluation did not complete; inspect its saved report before continuing.")
            live_candidate = await evaluator.aevaluate_batch(transcript_dir, candidate_dir,
                experiment_dir / "judge_runs", context_dir=context_dir)
            manifest["evaluation_runs"]["candidate"] = str(Path(live_candidate.output_dir) / "results.json")
            if any(result.execution_status != "completed" for result in live_candidate.results):
                raise RuntimeError("Candidate evaluation did not complete; inspect its saved report.")
        manifest["status"] = "evaluated_pending_human_review"
        print("Baseline report:", Path(live_baseline.output_dir) / "report.html")
        print("Candidate report:", Path(live_candidate.output_dir) / "report.html")
    except Exception as exc:
        manifest["status"] = "evaluation_failed"
        manifest["error_type"] = type(exc).__name__
        raise RuntimeError("Evaluation stopped. Inspect the saved run reports and manifest before retrying.") from None
    finally:
        save_manifest()
else:
    print("OFFLINE/default: judge evaluation skipped. No faithfulness or coverage scores were created.")

## 9. Compare genuine saved runs

Live results above are used when available. Otherwise set `SAVED_BASELINE_JSON` and `SAVED_CANDIDATE_JSON` to two existing result files and rerun from the top. Loading them makes no model requests.

The package refuses comparisons with different evaluator policies, judge settings, versions, meeting IDs, transcripts, or supplied context. Never relax those checks just to obtain a delta. Incomplete evaluations remain visible and do not enter aggregate deltas.

A saved pair may come from another experiment. Its recorded sources and judge settings control that comparison; loading it does not connect its summaries to the producer draft above.

In [ ]:
comparison = None
baseline_batch = live_baseline
candidate_batch = live_candidate
if baseline_batch is None and candidate_batch is None and (SAVED_BASELINE_JSON is not None or SAVED_CANDIDATE_JSON is not None):
    if SAVED_BASELINE_JSON is None or SAVED_CANDIDATE_JSON is None:
        raise ValueError("Select both baseline and candidate results.json paths.")
    saved_paths = [Path(SAVED_BASELINE_JSON), Path(SAVED_CANDIDATE_JSON)]
    if all(path.is_file() for path in saved_paths):
        baseline_batch, candidate_batch = [BatchResult.model_validate_json(path.read_text(encoding="utf-8")) for path in saved_paths]
    else:
        print("A selected results.json does not exist. Correct the paths; no comparison was fabricated.")
if baseline_batch is not None and candidate_batch is not None:
    comparison = compare_batches(baseline_batch, candidate_batch)
    comparison_path = experiment_dir / "comparison.json"
    comparison_path.write_text(json.dumps(comparison, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    write_csv(comparison["rows"], experiment_dir / "comparison.csv")
    manifest["comparison"] = {"path": str(comparison_path), "source": "live_experiment" if RUN_LIVE else "selected_saved_runs",
                              "baseline_run_id": baseline_batch.run_id, "candidate_run_id": candidate_batch.run_id,
                              "relationship_to_current_prompt_draft": "See recorded generation provenance" if RUN_LIVE else "Not established for externally saved runs"}
    save_manifest()
    print("Complete paired evaluations:", comparison["complete_pair_count"], "/", comparison["meeting_count"])
    for name, values in comparison["metric_deltas"].items():
        print(name, "— paired count:", values["paired_count"], "mean delta:", values["mean_delta"], "median delta:", values["median_delta"])
    print("Comparison saved:", comparison_path)
    display(Markdown("Positive deltas favor the candidate **for these recorded judgments**. Review material findings, not just score movement."))
else:
    print("No genuine saved result pair selected, so no quality scores or deltas are displayed.")
    print("After an evaluation, select its two results.json files above. Draft location:", experiment_dir)

## 10. Decide what changed, then test it elsewhere

Before looking at scores, have a human read the same sources and both summaries without the variant labels. Record the material issues, useful paraphrases, omissions, and uncertainty independently. Then reconcile the human review with the diagnostic findings.

For each paired case, record:

| Review question | Your note |
|---|---|
| Which material issue was actually fixed? | Cite the source and changed summary wording. |
| Which new material issue appeared? | Include regressions even when the mean score rose. |
| Which finding was a false alarm or a missed error? | Keep human evidence separate from judge suggestions. |
| Was the producer really changed? | Distinguish actual generated variants from a prewritten or manually edited control. |
| Is the result repeatable? | Repeat noisy cases under the same judge and inspect the variation. |
| Does the change generalize? | Test a held-out set that did not drive the edit; include long, noisy, and subtle financial cases. |

Do not tune to a single fictional meeting. Use the calibration guidance in `docs/calibration.md` for independent labels and representative cases. If you change judge rules, model settings, or methodology, establish a new baseline rather than treating it as the same producer experiment. A one-pair local test cannot establish production-wide or Zoom-specific improvement.

In [ ]:
print("Notebook completed in", "LIVE opt-in mode" if RUN_LIVE else "OFFLINE mode")
print("Producer candidate retained separately:", experiment_dir / "candidate_prompt.md")
print("Experiment manifest:", manifest_path)
print("Human review remains:", manifest["human_review"]["status"])
if not RUN_LIVE:
    print("Credentials were not loaded by this notebook; generation and judge calls were skipped.")
    print("Observed offline work: source inspection, exact producer prompt, candidate diff, and deterministic template checks.")
if comparison is None:
    print("No claim of score improvement has been made.")